# DVAnalysis Quickstart

This notebook demonstrates the core workflow of the `dvanalysis` library:

1. **Define** a stimulus protocol
2. **Load** DVA recordings from Imedos RTF exports
3. **Inspect** raw signals
4. **Denoise** with RPCA and Kotliar SMS (comparison)
5. **Extract** biomarkers per cycle
6. **Visualise** results

## Imports

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

from dvanalysis.domain import TimeWindow, StimulusCycle, StimulusProtocol
from dvanalysis.io import ImedosReader, DataReaderConfig
from dvanalysis.preprocessing import MyMethodRPCA, MyMethodConfig, KotliarPreprocessor, KotliarConfig
from dvanalysis.biomarkers import ParameterExtractor, ParameterExtractorConfig, make_default_definitions, make_paper_definitions
from dvanalysis.visualization import plot_triage_2d, plot_triage_2d_multi

## 1. Define the stimulus protocol

Standard 3-cycle DVA protocol used at TU Munich:
- 20 s global baseline
- 3 cycles of: 30 s baseline + 20 s flicker + 50 s recovery
- Total: 320 s

In [ ]:
protocol = StimulusProtocol(
    name="DVA_3cycle_custom",
    fs=25.0,
    description="20s global baseline + 3x(30s baseline + 20s flicker + 50s recovery)",
    global_baseline=TimeWindow("baseline", 0.0, 20.0),
    cycles=[
        StimulusCycle(index=0,
            baseline=TimeWindow("baseline",  20.0,  50.0),
            flicker= TimeWindow("flicker",   50.0,  70.0),
            recovery=TimeWindow("recovery",  70.0, 120.0)),
        StimulusCycle(index=1,
            baseline=TimeWindow("baseline", 120.0, 150.0),
            flicker= TimeWindow("flicker",  150.0, 170.0),
            recovery=TimeWindow("recovery", 170.0, 220.0)),
        StimulusCycle(index=2,
            baseline=TimeWindow("baseline", 220.0, 250.0),
            flicker= TimeWindow("flicker",  250.0, 270.0),
            recovery=TimeWindow("recovery", 270.0, 320.0)),
    ],
)

print(f"Protocol: {protocol.name}, fs={protocol.fs} Hz, {len(protocol.cycles)} cycles, duration={protocol.end_time_sec():.0f} s")

## 2. Load the dataset

The `ImedosReader` parses Imedos RTF/TXT locus exports and builds a `Dataset > Recording > Segment > SegmentSignal` hierarchy. Vessel type is inferred from the segment label (A → artery, V → vein).

In [ ]:
data_dir = Path("../../data/Healthy_Volunteers")

reader = ImedosReader(config=DataReaderConfig(protocol=protocol))
dataset = reader.read(data_dir)

print(f"Loaded {len(dataset.recordings)} recordings")
for rec in list(dataset.recordings)[:5]:
    segs = list(rec.segments.keys())
    print(f"  {rec.subject_id}_{rec.visit_id}: {segs}")

In [ ]:
# Pick one recording
rec = dataset.get_recording("183", "0")
print(rec.summary())

## 3. Inspect raw signals

The triage plot shows the raw median-over-loci trace (thin) with a running median smooth (thick), overlaid with protocol phase shading (baseline, flicker, recovery).

In [ ]:
# Multi-panel triage for all segments in this recording
plot_triage_2d_multi(rec, smooth_sec=2.0, show=True)

### 3D locus surface

Visualise the full (time x locus) matrix as a 3D surface. Each locus is one measurement point along the vessel — the surface shows how different loci respond to flicker stimulation.

In [ ]:
from dvanalysis.visualization import Plotter

plotter = Plotter()

# Artery A1 — 3D surface
seg_a1 = rec.segments["A1"]
plotter.plot_locus_surface_3d(seg_a1, title_prefix="Subject 183 — ", show=True)

## 4. Denoise with RPCA

The physiology-informed RPCA denoising pipeline:
1. Baseline-aware centering and MAD scaling
2. Masked smooth RPCA decomposition: signal = low-rank physiology (S) + sparse artefacts (A)
3. Locus aggregation with support gate
4. Harmonisation to % change from baseline

In [ ]:
rpca_config = MyMethodConfig(
    fill_missing=False,
    heartbeat_filter=False,
    standardize=True,
    rpca_lmb=0.55,                                              # sparsity penalty (lambda)
    rpca_gamma=1000.0,                                          # temporal smoothness
    rpca_max_iter=70,
    rpca_tol_rel=1e-3,
    locus_agg="median",
    harmonize_output=True,                                      # convert to % change from baseline
    harmonize_percent_mode="delta_over_baseline",
    harmonize_baseline_source="protocol_global",
    harmonize_aggregation_order="percent_then_aggregate",
    harmonize_baseline_per_locus=True,
    support_min_valid_frac=0.75,
    support_min_valid_abs=12,
    hampel_enable=True,
)

rpca = MyMethodRPCA(config=rpca_config)

# Run on all segments of this recording
rpca_results = {}
for seg_label, seg in rec.segments.items():
    rpca_results[seg_label] = rpca.run(seg.signal)
    r = rpca_results[seg_label]
    print(f"{seg_label} ({seg.vessel_type}): iter={r.diagnostics.get('n_iter', '?')}, "
          f"T={r.T}, P={r.P}, finite={np.sum(np.isfinite(r.s_hat))}/{r.T}")

### Kotliar SMS baseline (for comparison)

The Kotliar method aggregates loci (median), normalises per-cycle to baseline, applies a 4 s running median filter, and averages cycles into a template.

In [ ]:
kotliar = KotliarPreprocessor(config=KotliarConfig(
    mode="smooth_only",
    smooth_only_keep_outside_cycles=True,
))

kot_results = {}
for seg_label, seg in rec.segments.items():
    kot_results[seg_label] = kotliar.run(seg.signal)

print("Kotliar preprocessing done for all segments.")

### Compare RPCA vs Kotliar — interactive Plotly traces

In [ ]:
seg_label = "A1"
seg = rec.segments[seg_label]
sig = seg.signal
rpca_r = rpca_results[seg_label]
kot_r = kot_results[seg_label]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=[f"Raw signal — {seg_label} ({seg.vessel_type})",
                                   f"Denoised — RPCA vs Kotliar SMS"])

# Raw median trace
y_raw = sig.aggregate_over_loci(agg="median")
fig.add_trace(go.Scatter(x=sig.t, y=y_raw, mode="lines", name="Raw (median over loci)",
                         line=dict(width=0.8, color="#999999"), opacity=0.7), row=1, col=1)

# Denoised traces
fig.add_trace(go.Scatter(x=sig.t, y=rpca_r.s_hat, mode="lines", name="RPCA",
                         line=dict(width=2, color="#B2182B")), row=2, col=1)
fig.add_trace(go.Scatter(x=sig.t, y=kot_r.s_hat, mode="lines", name="Kotliar SMS",
                         line=dict(width=1.5, color="#2166AC"), opacity=0.8), row=2, col=1)

# Flicker shading
for cyc in protocol.cycles:
    for row in [1, 2]:
        fig.add_vrect(x0=cyc.flicker.start_sec, x1=cyc.flicker.end_sec,
                      fillcolor="rgba(255,215,0,0.15)", line_width=0, row=row, col=1)

fig.update_yaxes(title_text="Diameter (a.u.)", row=1, col=1)
fig.update_yaxes(title_text="% change from baseline", row=2, col=1)
fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.update_layout(template="plotly_dark", height=600, width=1100, legend=dict(x=0.01, y=0.48))
fig.show()

### All four segments — RPCA denoised traces

In [ ]:
segment_order = [sl for sl in ["A1", "A2", "V3", "V4"] if sl in rec.segments]
colours = {"A1": "#B2182B", "A2": "#E08080", "V3": "#2166AC", "V4": "#67A9CF"}

fig = make_subplots(rows=len(segment_order), cols=1, shared_xaxes=True,
                    vertical_spacing=0.04,
                    subplot_titles=[f"{sl} ({rec.segments[sl].vessel_type})" for sl in segment_order])

for i, sl in enumerate(segment_order, 1):
    r = rpca_results[sl]
    t = rec.segments[sl].signal.t
    fig.add_trace(go.Scatter(x=t, y=r.s_hat, mode="lines", name=sl,
                             line=dict(width=1.5, color=colours.get(sl, "#888")),
                             showlegend=(i == 1)), row=i, col=1)
    fig.update_yaxes(title_text="% bsl", row=i, col=1)

    for cyc in protocol.cycles:
        fig.add_vrect(x0=cyc.flicker.start_sec, x1=cyc.flicker.end_sec,
                      fillcolor="rgba(255,215,0,0.12)", line_width=0, row=i, col=1)

fig.update_xaxes(title_text="Time (s)", row=len(segment_order), col=1)
fig.update_layout(template="plotly_dark", height=250*len(segment_order), width=1100,
                  title_text=f"Subject {rec.subject_id} — RPCA denoised traces")
fig.show()

## 5. Extract biomarkers

Biomarkers are extracted per cycle. The default set includes:
- **Max dilation**: peak value during flicker
- **Max constriction**: minimum value during recovery
- **Dilation amplitude**: max dilation minus max constriction

In [ ]:
extractor = ParameterExtractor(
    config=ParameterExtractorConfig(trace_source="s_hat", strict=False),
    definitions=make_default_definitions(),
)

all_rows = []
for seg_label, seg in rec.segments.items():
    rows = extractor.analyze_segment(seg, rpca_results[seg_label], method_name="rpca")
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)
display_cols = ["segment_label", "vessel_type", "cycle_index", "max_dilation", "max_constriction", "dilation_amplitude"]
df[display_cols].round(3)

### Extended paper feature set

The `make_paper_definitions()` set adds timing features (time to max dilation/constriction), integrated response (AUC), and flicker-induced change — all with per-cycle baseline correction.

In [ ]:
paper_extractor = ParameterExtractor(
    config=ParameterExtractorConfig(trace_source="s_hat", strict=False),
    definitions=make_paper_definitions(),
)

paper_rows = []
for seg_label, seg in rec.segments.items():
    rows = paper_extractor.analyze_segment(seg, rpca_results[seg_label], method_name="rpca")
    paper_rows.extend(rows)

df_paper = pd.DataFrame(paper_rows)
bio_cols = list(make_paper_definitions().keys())
df_paper[["segment_label", "cycle_index"] + bio_cols].round(3)

## 6. Per-cycle overlay

Align each cycle to flicker onset and overlay — a standard DVA visualisation showing the vascular response shape.

In [ ]:
from dvanalysis.visualization.cycles import extract_cycle_trace

seg_label = "A1"
rpca_r = rpca_results[seg_label]
t_abs = rec.segments[seg_label].signal.t

PRE, POST = 15, 50
t_common = np.arange(-PRE, POST, 1.0 / protocol.fs)
cycle_colours = ["#E08080", "#B2182B", "#800000"]

fig = go.Figure()

# Flicker shading
fig.add_vrect(x0=0, x1=20, fillcolor="rgba(255,215,0,0.12)", line_width=0)
fig.add_hline(y=0, line_dash="dot", line_color="#666666", line_width=0.5)

for i, cyc in enumerate(protocol.cycles):
    trace = extract_cycle_trace(rpca_r.s_hat, t_abs, cyc.flicker.start_sec,
                                pre_sec=PRE, post_sec=POST, t_common=t_common)
    if trace is not None:
        fig.add_trace(go.Scatter(
            x=t_common, y=trace, mode="lines",
            name=f"Cycle {i+1}",
            line=dict(width=2, color=cycle_colours[i]),
        ))

fig.update_layout(
    template="plotly_dark", width=900, height=400,
    title=f"Subject {rec.subject_id} — {seg_label} per-cycle overlay (RPCA)",
    xaxis_title="Time relative to flicker onset (s)",
    yaxis_title="% change from baseline",
)
fig.show()

### Method comparison — 3-cycle overlay

Compare raw, Kotliar SMS, and RPCA on the same segment: each method's 3 individual cycles (faint) plus the median cycle (bold), all aligned to flicker onset.

In [ ]:
from dvanalysis.visualization import plot_three_cycles_method_comparison

# Plot for A1
seg_label = "A1"
sig = rec.segments[seg_label].signal

fig = plot_three_cycles_method_comparison(
    sig,
    methods={"kotliar": kot_results[seg_label], "rpca": rpca_results[seg_label]},
    include_raw=True,
    title=f"Subject {rec.subject_id} — {seg_label} ({rec.segments[seg_label].vessel_type})",
)
fig.show()

In [ ]:
# All four segments
for sl in ["A1", "A2", "V3", "V4"]:
    if sl not in rec.segments:
        continue
    fig = plot_three_cycles_method_comparison(
        rec.segments[sl].signal,
        methods={"kotliar": kot_results[sl], "rpca": rpca_results[sl]},
        include_raw=True,
        title=f"Subject {rec.subject_id} — {sl} ({rec.segments[sl].vessel_type})",
        height=450,
    )
    fig.show()

## 7. Using the Pipeline (one-liner)

For quick analysis, the `Pipeline` class composes IO + preprocessing + biomarker extraction into a single call.

In [ ]:
from dvanalysis import Pipeline

pipe = Pipeline(preprocessor="rpca", preprocessor_config=rpca_config)
output = pipe.run(str(data_dir), protocol=protocol)

print(f"Pipeline output:")
print(f"  Recordings: {len(output['dataset'].recordings)}")
print(f"  Preprocessed segments: {len(output['results'])}")
print(f"  Parameter rows: {len(output['parameters'])}")
output["parameters"][["segment_label", "vessel_type", "cycle_index",
                       "max_dilation", "max_constriction", "dilation_amplitude"]].head(12).round(3)